# Notebook 4.2: Cleaning Real-World Data

**Companion to Chapter 4: Implementing Data Pre-processing in Python**  
*Machine Learning with Python: Principles and Practical Techniques*

> **Estimated time:** 40–50 minutes  
> **Level:** Beginner  
> **Environment:** Google Colab or Jupyter Notebook

---

## Related chapter ideas

This notebook develops the data-cleaning phase of pre-processing. It continues with the synthetic student-success dataset introduced in Notebook 4.1.

## Learning objectives

By the end of this notebook, you will be able to:

1. create a systematic data-quality audit;
2. detect and remove exact duplicate records;
3. identify and handle missing numerical values;
4. standardize inconsistent categorical labels;
5. distinguish unusual observations from definite errors;
6. validate a cleaned dataset using explicit rules; and
7. document cleaning decisions for reproducibility and responsible use.


## What will you build?

You will turn a deliberately imperfect dataset into a clean, validated table. Rather than applying cleaning commands mechanically, you will follow a defensible workflow:

**Profile → Diagnose → Decide → Transform → Validate → Document**

> **Key principle:** Cleaning is not the same as deleting inconvenient data. Every change should have a reason, and the original data should remain available.


## 1. Import the libraries


In [ ]:
from io import StringIO

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 2)

print("Pandas version:", pd.__version__)


## 2. Load the same dataset used in Notebook 4.1


In [ ]:
student_csv = """student_id,study_hours,attendance_pct,previous_score,learning_mode,programming_experience,assignments_submitted,final_score,passed
S001,5.5,92,78,In-person,Beginner,9,84,Yes
S002,3.0,75,65,Online,No prior experience,7,68,Yes
S003,1.5,61,58,Hybrid,No prior experience,5,55,No
S004,6.0,95,88,In-person,Intermediate,10,91,Yes
S005,2.0,70,62,Online,Beginner,6,63,Yes
S006,4.5,85,74,Hybrid,Beginner,8,79,Yes
S007,1.0,55,51,Online,No prior experience,4,48,No
S008,7.0,98,91,In-person,Advanced,10,95,Yes
S009,3.5,82,69,Hybrid,Beginner,8,73,Yes
S010,2.5,67,60,Online,No prior experience,6,59,No
S011,5.0,90,81,In-person,Intermediate,9,86,Yes
S012,4.0,88,76,Hybrid,Beginner,8,80,Yes
S013,2.0,,57,Online,No prior experience,5,54,No
S014,6.5,96,89,In-person,Advanced,10,93,Yes
S015,3.0,78,,Hybrid,Beginner,7,70,Yes
S016,1.5,63,55,Online,No prior experience,4,52,No
S017,5.5,91,83,In-person,Intermediate,9,88,Yes
S018,4.0,84,72,hybrid,Beginner,8,77,Yes
S019,2.5,72,64,Online,Beginner,6,65,Yes
S020,6.0,94,86,In-person,Advanced,10,90,Yes
S021,3.5,80,70,Hybrid,Beginner,7,72,Yes
S022,1.0,58,49,Online,No prior experience,3,45,No
S023,4.5,87,75,In-person,Intermediate,9,82,Yes
S024,2.0,69,59,Online,No prior experience,5,57,No
S025,5.0,89,80,Hybrid,Intermediate,9,85,Yes
S026,3.0,76,67,Online,Beginner,7,69,Yes
S027,6.5,97,90,In-person,Advanced,10,94,Yes
S028,1.5,60,53,Online,No prior experience,4,50,No
S029,4.0,83,73,Hybrid,Beginner,8,78,Yes
S030,2.5,74,63,Online,Beginner,6,64,Yes
S030,2.5,74,63,Online,Beginner,6,64,Yes
"""

students = pd.read_csv(StringIO(student_csv))
print("Dataset loaded successfully.")


In [ ]:
# Preserve an untouched copy before making any changes.
raw_students = students.copy(deep=True)
clean_students = students.copy(deep=True)

print("Working copy created:", clean_students.shape)
display(clean_students.head())


### Why keep the raw data?

An untouched copy supports auditing, reproducibility, and recovery from an incorrect transformation. In larger projects, raw data is usually stored separately and treated as read-only.


## 3. Build a reusable quality report


In [ ]:
def quality_report(dataframe):
    # Return a compact column-level data-quality report.
    return pd.DataFrame({
        "dtype": dataframe.dtypes.astype(str),
        "missing": dataframe.isna().sum(),
        "missing_pct": dataframe.isna().mean().mul(100).round(1),
        "unique": dataframe.nunique(dropna=True),
    })


display(quality_report(clean_students))
print("Exact duplicate rows:", clean_students.duplicated().sum())


In [ ]:
# Inspect the observed labels in categorical columns.
categorical_columns = clean_students.select_dtypes(include="object").columns

for column in categorical_columns:
    print(f"{column}: {clean_students[column].dropna().unique().tolist()}")


### Initial diagnosis

The audit reveals three known problems:

- one duplicate record;
- missing values in `attendance_pct` and `previous_score`; and
- inconsistent capitalization: `Hybrid` versus `hybrid`.

We will address one problem at a time and verify each result.


## 4. Detect and remove duplicate records


In [ ]:
duplicate_mask = clean_students.duplicated(keep=False)
duplicate_records = clean_students.loc[duplicate_mask].sort_values("student_id")

display(duplicate_records)


`duplicated(keep=False)` marks every member of a duplicate group. This is useful for inspection because it shows both the original and repeated records.

Before removal, ask whether the records are truly duplicates. Two identical transactions may represent legitimate separate events, but two identical student rows with the same identifier are more likely an accidental repetition.


In [ ]:
rows_before = len(clean_students)
clean_students = clean_students.drop_duplicates().reset_index(drop=True)
rows_after = len(clean_students)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows removed:", rows_before - rows_after)
assert clean_students.duplicated().sum() == 0


## 5. Diagnose missing values


In [ ]:
missing_rows = clean_students.loc[
    clean_students.isna().any(axis=1),
    ["student_id", "attendance_pct", "previous_score", "final_score", "passed"],
]

display(missing_rows)


### Missing does not mean zero

A missing attendance value does not mean 0% attendance, and a missing previous score does not mean a score of zero. Replacing missing values with zero would introduce information that the dataset never contained.

Common strategies include:

- recovering the value from a trusted source;
- removing a row or column when justified;
- imputing with a representative value; or
- using a model that can handle missing values.

The correct decision depends on why the value is missing and how the data will be used.


In [ ]:
# Compare mean and median before choosing an imputation value.
numeric_missing_columns = ["attendance_pct", "previous_score"]

imputation_options = clean_students[numeric_missing_columns].agg(["mean", "median"]).T
display(imputation_options)


For this learning activity, we will use **median imputation**. The median is less sensitive than the mean to unusually high or low values.

> **Important modeling note:** Here we are producing a cleaned exploratory dataset. When building a predictive model, imputation values must be learned from the **training data only**. Notebook 4.3 will use a Scikit-learn pipeline to prevent data leakage.


In [ ]:
imputation_log = {}

for column in numeric_missing_columns:
    median_value = clean_students[column].median()
    missing_count = clean_students[column].isna().sum()
    clean_students[column] = clean_students[column].fillna(median_value)
    imputation_log[column] = {
        "strategy": "median",
        "value": median_value,
        "values_replaced": int(missing_count),
    }

pd.DataFrame(imputation_log).T


In [ ]:
print("Remaining missing values:", clean_students.isna().sum().sum())
assert clean_students[numeric_missing_columns].isna().sum().sum() == 0


## 6. Standardize categorical labels


In [ ]:
print("Before:", clean_students["learning_mode"].value_counts().to_dict())

clean_students["learning_mode"] = (
    clean_students["learning_mode"]
    .str.strip()
    .str.title()
)

print("After:", clean_students["learning_mode"].value_counts().to_dict())


String standardization can combine labels that differ only in whitespace or capitalization. However, automated corrections should be reviewed. For example, abbreviations and product names may require a controlled mapping instead of `.str.title()`.


In [ ]:
# Convert stable categorical columns to the category dtype.
category_columns = ["learning_mode", "programming_experience", "passed"]

for column in category_columns:
    clean_students[column] = clean_students[column].astype("category")

display(clean_students.dtypes.to_frame("cleaned_dtype"))


## 7. Check ranges and possible outliers


In [ ]:
range_checks = pd.DataFrame({
    "observed_min": clean_students[[
        "study_hours", "attendance_pct", "previous_score",
        "assignments_submitted", "final_score"
    ]].min(),
    "observed_max": clean_students[[
        "study_hours", "attendance_pct", "previous_score",
        "assignments_submitted", "final_score"
    ]].max(),
    "expected_min": [0, 0, 0, 0, 0],
    "expected_max": [168, 100, 100, 10, 100],
})

display(range_checks)


Range checks use domain rules. Attendance and scores should fall between 0 and 100, while submitted assignments cannot exceed the number assigned. A value can be statistically unusual yet still valid, or statistically ordinary yet logically impossible.


In [ ]:
def iqr_outliers(series):
    # Return values outside 1.5 times the interquartile range.
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series[(series < lower) | (series > upper)], lower, upper


numeric_columns = [
    "study_hours", "attendance_pct", "previous_score",
    "assignments_submitted", "final_score"
]

for column in numeric_columns:
    flagged, lower, upper = iqr_outliers(clean_students[column])
    print(f"{column}: {len(flagged)} flagged | interval [{lower:.2f}, {upper:.2f}]")


In [ ]:
clean_students[numeric_columns].plot(
    kind="box",
    subplots=True,
    layout=(2, 3),
    figsize=(11, 6),
    sharex=False,
    sharey=False,
    color="#4C78A8",
)
plt.suptitle("Boxplots for Numerical Variables", y=1.02)
plt.tight_layout()
plt.show()


### A controlled outlier experiment

The current data contains no strong IQR outlier in `study_hours`. To understand the method, we will create a temporary copy and simulate a possible entry error. The cleaned dataset itself will not be changed.


In [ ]:
what_if = clean_students.copy()
what_if.loc[0, "study_hours"] = 25.0

flagged, lower, upper = iqr_outliers(what_if["study_hours"])
print(f"IQR interval: [{lower:.2f}, {upper:.2f}]")
display(what_if.loc[flagged.index, ["student_id", "study_hours"]])


A value of 25 study hours is unusual, but it is not impossible. It should be investigated—not automatically deleted. If the source confirms that it was meant to be `2.5`, correcting it is appropriate. Without evidence, removal could erase a legitimate observation.


## 8. Validate the cleaned dataset


In [ ]:
expected_modes = {"In-Person", "Online", "Hybrid"}
expected_experience = {"No prior experience", "Beginner", "Intermediate", "Advanced"}

validation_results = {
    "No duplicate rows": clean_students.duplicated().sum() == 0,
    "No missing values": clean_students.isna().sum().sum() == 0,
    "Attendance within 0–100": clean_students["attendance_pct"].between(0, 100).all(),
    "Scores within 0–100": clean_students[["previous_score", "final_score"]]
        .apply(lambda column: column.between(0, 100).all()).all(),
    "Assignments within 0–10": clean_students["assignments_submitted"].between(0, 10).all(),
    "Learning modes valid": set(clean_students["learning_mode"].astype(str)) <= expected_modes,
    "Experience levels valid": set(clean_students["programming_experience"].astype(str)) <= expected_experience,
    "Student IDs unique": clean_students["student_id"].is_unique,
}

validation_table = pd.Series(validation_results, name="passed").to_frame()
display(validation_table)
assert validation_table["passed"].all(), "One or more validation checks failed."


### Why validation matters

Code can run successfully while producing incorrect data. Validation converts expectations into explicit, repeatable checks. If new data violates a rule, the workflow should stop and request investigation.


## 9. Compare before and after


In [ ]:
comparison = pd.DataFrame({
    "raw": {
        "rows": len(raw_students),
        "duplicate_rows": int(raw_students.duplicated().sum()),
        "missing_values": int(raw_students.isna().sum().sum()),
        "learning_mode_labels": raw_students["learning_mode"].nunique(),
    },
    "cleaned": {
        "rows": len(clean_students),
        "duplicate_rows": int(clean_students.duplicated().sum()),
        "missing_values": int(clean_students.isna().sum().sum()),
        "learning_mode_labels": clean_students["learning_mode"].nunique(),
    },
})

display(comparison)


In [ ]:
cleaning_log = pd.DataFrame([
    {
        "issue": "Exact duplicate row",
        "decision": "Removed one repeated S030 record",
        "reason": "Same identifier and identical values",
    },
    {
        "issue": "Missing attendance_pct",
        "decision": f"Replaced with median ({imputation_log['attendance_pct']['value']:.1f})",
        "reason": "Retained row; median is robust to extreme values",
    },
    {
        "issue": "Missing previous_score",
        "decision": f"Replaced with median ({imputation_log['previous_score']['value']:.1f})",
        "reason": "Retained row; median is robust to extreme values",
    },
    {
        "issue": "Hybrid/hybrid inconsistency",
        "decision": "Standardized capitalization",
        "reason": "Labels represent the same category",
    },
])

display(cleaning_log)


## 10. Preview or export the cleaned data


In [ ]:
display(clean_students.head())

# Convert the cleaned table to CSV text without creating an external file.
clean_csv_preview = clean_students.to_csv(index=False)
print(clean_csv_preview[:500])


To save the cleaned dataset in your own Colab session, run:

```python
clean_students.to_csv("student_success_clean.csv", index=False)
```

Keep the cleaning code and decision log with the exported data so another person can reproduce the result.


## 11. Guided practice


Using `raw_students`, complete these tasks:

1. Count duplicated `student_id` values using `duplicated(subset=...)`.
2. Display only rows containing at least one missing value.
3. Create a standardized version of `passed` using `.str.strip().str.title()`.
4. Confirm that all `final_score` values lie between 0 and 100.


In [ ]:
# Write your solution here.


<details>
<summary><strong>Open the suggested solution</strong></summary>

```python
# 1. Duplicate identifiers
print(raw_students.duplicated(subset="student_id").sum())

# 2. Rows with at least one missing value
display(raw_students.loc[raw_students.isna().any(axis=1)])

# 3. Standardized passed labels
standardized_passed = raw_students["passed"].str.strip().str.title()
display(standardized_passed.value_counts())

# 4. Final-score range check
print(raw_students["final_score"].between(0, 100).all())
```

</details>


## 12. Challenge: Design a cleaning policy


Imagine that a future data file contains these new problems:

- `attendance_pct = 108`;
- `learning_mode = "HYBRID "`;
- two rows share a `student_id` but have different final scores; and
- 35% of `previous_score` values are missing.

For each problem, decide whether you would **correct, standardize, investigate, impute, remove, or retain** the data. Explain what evidence you would need. There may be more than one defensible answer.


## 13. Responsible data cleaning


Cleaning decisions can change who is represented in a dataset. Before removing records or filling values, ask:

- Are missing values concentrated in a particular group?
- Would row deletion systematically exclude some learners?
- Does an outlier represent an error or an important real experience?
- Could imputation hide unequal access, participation, or opportunity?
- Can another person understand and reproduce every transformation?

For high-impact uses, conduct subgroup analysis, consult domain experts, protect privacy, and preserve meaningful human review.


## 14. Reflection


1. Why should raw data remain unchanged?
2. When is median imputation preferable to mean imputation?
3. Why should an outlier be investigated before removal?
4. What is the difference between a statistical outlier and an invalid value?
5. Why must imputation be fitted only on training data during predictive modeling?
6. Which cleaning decision in this notebook could most affect downstream results, and why?


## 15. Key takeaways


- Begin cleaning with a systematic profile of structure, missingness, duplicates, and categories.
- Preserve an untouched raw copy and document every transformation.
- Missing values require context; they are not automatically zero.
- Duplicate detection should be followed by investigation before removal.
- Standardize categorical labels carefully and verify the resulting categories.
- Statistical outliers are signals for investigation, not automatic deletion targets.
- Validation rules convert assumptions into executable quality checks.
- During modeling, learn preprocessing parameters from training data only.

### Looking ahead

In **Notebook 4.3: Preparing Features for Machine Learning**, you will split data correctly and use Scikit-learn pipelines for imputation, categorical encoding, normalization, and standardization without data leakage.
